# Advanced Wave Dynamics

## Level 1 Basic 2D wave equation
This guide explains how to implement a 2D wave equation solver using PhysicsNeMo. The wave equation is a second-order partial differential equation that describes the propagation of waves, such as sound waves, water waves, or electromagnetic waves.

## Mathematical Formulation
The 2D wave equation is given by:

$$\begin{align*} u_{tt} &= c^2 u_{xx} + c^2 u_{yy}
\end{align*}
$$

where:
- u(x,t) is the wave amplitude
- c is the wave speed
- x is the spatial coordinate
- t is time

## Implementation Steps

### 1. Import Required Libraries
```python
import os
import warnings

import numpy as np
from sympy import Symbol, sin

import physicsnemo.sym
from physicsnemo.sym.hydra import to_yaml, to_absolute_path, instantiate_arch, PhysicsNeMoConfig
from physicsnemo.sym.hydra.utils import compose
from physicsnemo.sym.hydra.config import PhysicsNeMoConfig

from physicsnemo.sym.solver import Solver
from physicsnemo.sym.domain import Domain
from physicsnemo.sym.geometry.primitives_2d import Rectangle
from physicsnemo.sym.domain.constraint import (
    PointwiseBoundaryConstraint,
    PointwiseInteriorConstraint,
)

from physicsnemo.sym.domain.validator import PointwiseValidator
from physicsnemo.sym.key import Key
from physicsnemo.sym.node import Node

from sympy import Symbol, Function, Number
from physicsnemo.sym.eq.pde import PDE
from physicsnemo.sym.utils.io import ValidatorPlotter

import matplotlib.pyplot as plt
plt.rcParams['image.cmap'] = 'jet'
```

### 2. Define the Wave Equation Class
Create a class that inherits from PDE to define the wave equation:
- $x$ and $t$ are symbolic variables representing spatial and temporal coordinates. 
- These variables are bundled into a `input_variables` dictionary. 
- A function $u$ is created, representing a wave function in terms of x and t. 
- The input c is the wave speed coefficient, treated as a function if it's a string, or a number if it's a float or int. 
- A wave equation is formulated and stored in the equations dictionary 
- `diff` is used to compute the derivates

```python
class WaveEquation2D(PDE):
    name = "WaveEquation2D"

    def __init__(self, c=1.0):
        # Define coordinates
        x = Symbol("x")
        y = Symbol("y")
        t = Symbol("t")
        
        # Define input variables
        input_variables = {"x": x, "y": y, "t": t}
        
        # Define the wave function
        u = Function("u")(*input_variables)
        
        # Set wave speed coefficient
        if type(c) is str:
            c = Function(c)(*input_variables)
        elif type(c) in [float, int]:
            c = Number(c)
        
        # Define the wave equation
        self.equations = {}
        self.equations["wave_equation"] = FIXME # Fill in: Add wave equation here
```

### 3. Set Up the Solver
Create a main function to set up and run the solver:
- Define the geometry, equations, instantiate network architectures, and nodes in PhysicsNeMo
- Define the initial condition constraint
$$u(x,y,0) = \sin(x)\sin(y), u_t(x,y,0)= \sin(x)\sin(y)$$
- Define the boundary constraint
$$ u(0,y,t)=u(\pi,y,t)=0$$
$$ u(x,0,t)=u(x,\pi,t)=0$$
- Define the PDE constraint
$$ u_{tt} = c^2 u_{xx} + c^2 u_{yy}$$
and set $c=1.0$
- Add validator with exact solution
$$u(x,y,t) = \sin(x)\sin(y)(\sin(t)+\cos(t))$$
```python
@physicsnemo.sym.main(config_path="conf", config_name="config_wave")
def run(cfg: PhysicsNeMoConfig) -> None:
    # Initialize wave equation with wave speed 1.0
    c = FIXME # Fill in: c value
    we = WaveEquation2D(c=c)
    
    # Create neural network
    wave_net = instantiate_arch(
        input_keys=[Key("x"), Key("y"), Key("t")],
        output_keys=[Key("u")],
        cfg=cfg.arch.fully_connected,
    )
    
    # Create nodes for the solver
    nodes = we.make_nodes() + [wave_net.make_node(name="wave_network")]
    
    # Define geometry and time range
    x, y, t_symbol = Symbol("x"), Symbol("y"), Symbol("t")
    L = float(np.pi)
    geo = Rectangle((0, 0), (L, L))
    time_range = {t_symbol: (0, 2 * L)}
    
    # Create domain
    domain = Domain()
    
    # Add initial condition
    IC = PointwiseInteriorConstraint(
        nodes=nodes,
        geometry=geo,
        outvar={ FIXME }, # Fill in: Add initial condition here
        batch_size=cfg.batch_size.IC,
        lambda_weighting={"u": 1.0, "u__t": 1.0},
        parameterization={t_symbol: 0.0},
    )
    domain.add_constraint(IC, "IC")
    
    # Add boundary condition
    BC = PointwiseBoundaryConstraint(
        nodes=nodes,
        geometry=geo,
        outvar={ FIXME }, # Fill in: Add boundary condition here
        lambda_weighting={"u": 1.0},
        batch_size=cfg.batch_size.BC,
        parameterization=time_range,
    )
    domain.add_constraint(BC, "BC")
    
    # Add interior constraint
    interior = PointwiseInteriorConstraint(
        nodes=nodes,
        geometry=geo,
        outvar={ FIXME }, # Fill in: Add PDE constraint here
        batch_size=cfg.batch_size.interior,
        parameterization=time_range,
    )
    domain.add_constraint(interior, "interior")
    
    # Add validation data
    deltaT = 0.1
    deltaX = 0.1
    deltaY = 0.1
    x_vals = np.arange(0, L, deltaX)
    y_vals = np.arange(0, L, deltaY)
    t_vals = np.arange(0, L/4, deltaT)
    X, Y, T = np.meshgrid(x_vals, y_vals, t_vals)
    X = np.expand_dims(X.flatten(), axis=-1)
    Y = np.expand_dims(Y.flatten(), axis=-1)
    T = np.expand_dims(T.flatten(), axis=-1)
    u = # add formula of exact solution
    invar_numpy = {"x": X, "y": Y, "t": T}
    outvar_numpy = {"u": u}
    
    # Create validator
    validator = PointwiseValidator(
        nodes=nodes,
        invar=invar_numpy,
        true_outvar=outvar_numpy,
        batch_size=128,
        plotter=ValidatorPlotter(),
    )
    domain.add_validator(validator)
    
    # Create and run solver
    slv = Solver(cfg, domain)
    slv.solve()
```

1. Fill in all missing parts
2. Place the configuration file in the conf directory
3. Run the script:

In [ ]:
!python wave_l1.py

## Level 2 Variable Wave Speed

In Level 2, we extend the wave equation to include spatially-varying wave speed $c(x, y)$. This is more realistic for many physical scenarios where the medium properties change across space (e.g., seismic waves in heterogeneous earth, acoustic waves in varying media).

### Mathematical Formulation
The 2D wave equation with variable wave speed is:

$$\begin{align*} u_{tt} &= c^2(x,y) u_{xx} + c^2(x,y) u_{yy}
\end{align*}
$$

where $c(x,y)$ is now a function of spatial coordinates.

### Problem Setup

For this level, we'll use a spatially-varying wave speed:
$$c(x, y) = 1.0 + 0.5 \sin(x) \cos(y)$$

This creates regions where waves propagate at different speeds.

**Modified Initial Conditions:**
$$u(x,y,0) = \sin(x)\sin(y), \quad u_t(x,y,0) = 0$$

**Boundary Conditions** (same as Level 1):
$$u(0,y,t) = u(\pi,y,t) = 0$$
$$u(x,0,t) = u(x,\pi,t) = 0$$

**Note:** Unlike Level 1, we do NOT have a closed-form exact solution for variable wave speed. We will use a reference numerical solution or train without analytical validation.

### Implementation Steps for Level 2

#### 1. Modify the Wave Equation Class

You need to update the `WaveEquation2D` class to handle variable wave speed as a function:

```python
class WaveEquation2DVarSpeed(PDE):
    name = "WaveEquation2DVarSpeed"

    def __init__(self, c="c"):
        # Define coordinates
        x = Symbol("x")
        y = Symbol("y")
        t = Symbol("t")
        
        # Define input variables
        input_variables = {"x": x, "y": y, "t": t}
        
        # Define the wave function
        u = Function("u")(*input_variables)
        
        # Define variable wave speed as a function
        if type(c) is str:
            c = Function(c)(*input_variables)
        elif type(c) in [float, int]:
            c = Number(c)
        
        # Define the wave equation with variable c
        self.equations = {}
        self.equations["wave_equation"] = FIXME # Fill in: same as Level 1
```

#### 2. Create a Neural Network for Wave Speed

Since $c(x,y)$ is spatially varying, we need to provide it to the network. We can either:
- **Option A**: Use an analytical expression for $c(x,y)$ as a Node
- **Option B**: Learn $c(x,y)$ as part of the inverse problem (advanced)

For this challenge, we'll use **Option A** with the analytical expression.

```python
@physicsnemo.sym.main(config_path="conf", config_name="config_wave")
def run(cfg: PhysicsNeMoConfig) -> None:
    # Initialize wave equation with variable wave speed
    we = WaveEquation2DVarSpeed(c="c")
    
    # Create neural network for u
    wave_net = instantiate_arch(
        input_keys=[Key("x"), Key("y"), Key("t")],
        output_keys=[Key("u")],
        cfg=cfg.arch.fully_connected,
    )
    
    # Define wave speed function c(x, y)
    # c(x,y) = 1.0 + 0.5*sin(x)*cos(y)
    x, y, t_symbol = Symbol("x"), Symbol("y"), Symbol("t")
    
    # Create a node for the wave speed function
    c_node = Node.from_sympy(
        FIXME, # Fill in: create expression for c using sin and Symbol
    )
    
    # Create nodes for the solver
    nodes = we.make_nodes() + [wave_net.make_node(name="wave_network"), c_node]
    
    # Define geometry and time range
    L = float(np.pi)
    geo = Rectangle((0, 0), (L, L))
    time_range = {t_symbol: (0, 2 * L)}
    
    # Create domain
    domain = Domain()
    
    # Add initial condition (modified for Level 2)
    IC = PointwiseInteriorConstraint(
        nodes=nodes,
        geometry=geo,
        outvar={ FIXME }, # Fill in: Add two initial conditions here 
        batch_size=cfg.batch_size.IC,
        lambda_weighting={"u": 1.0, "u__t": 1.0},
        parameterization={t_symbol: 0.0},
    )
    domain.add_constraint(IC, "IC")
    
    # Add boundary condition (same as Level 1)
    BC = PointwiseBoundaryConstraint(
        nodes=nodes,
        geometry=geo,
        outvar={ FIXME }, # Fill in: same as Level 1
        lambda_weighting={"u": 1.0},
        batch_size=cfg.batch_size.BC,
        parameterization=time_range,
    )
    domain.add_constraint(BC, "BC")
    
    # Add interior constraint (PDE)
    interior = PointwiseInteriorConstraint(
        nodes=nodes,
        geometry=geo,
        outvar={ FIXME }, # Fill in: same as Level 1
        batch_size=cfg.batch_size.interior,
        parameterization=time_range,
    )
    domain.add_constraint(interior, "interior")
    
    # Note: For Level 2, we skip analytical validation since no closed-form solution exists
    # You can add monitoring points or compare with numerical reference if available
    
    # Create and run solver
    slv = Solver(cfg, domain)
    slv.solve()
```

1. Fill in all missing parts
2. Place the configuration file in the conf directory
3. Run the script:

In [ ]:
!python wave_l2.py

## Level 3 Complex Boundaries and Circular Domain

In Level 3, we move beyond simple rectangular geometries to explore wave propagation in more complex domains. This level introduces:
- **Circular domain** instead of rectangular
- **Robin boundary conditions** (mixed Dirichlet-Neumann) to model partial reflection
- **Multiple point sources** for richer wave patterns

This is more realistic for applications like acoustic wave scattering, radar systems, or seismic wave propagation around obstacles.

### Mathematical Formulation

We return to the constant wave speed equation but with a circular domain:

$$\begin{align*} u_{tt} &= c^2 (u_{xx} + u_{yy}), \quad (x,y) \in \Omega
\end{align*}
$$

where $\Omega$ is a circular domain: $x^2 + y^2 \leq R^2$ with radius $R = 1.0$.

### Problem Setup

**Domain**: Circle of radius $R = 1.0$ centered at origin

**Wave Speed**: $c = 1.0$ (constant)

**Initial Conditions** (two point sources):
$$u(x,y,0) = e^{-20((x-0.3)^2 + y^2)} + e^{-20((x+0.3)^2 + y^2)}$$
$$u_t(x,y,0) = 0$$

This creates two Gaussian bumps that propagate outward and interact.

**Robin Boundary Conditions** (partial reflection at $r = R$):
$$\alpha u + \beta \frac{\partial u}{\partial n} = 0, \quad \text{on } \partial\Omega$$

where $\frac{\partial u}{\partial n}$ is the normal derivative, and we use $\alpha = 1.0$, $\beta = 0.5$.

This allows partial wave transmission through the boundary (more realistic than hard walls).

### Implementation Steps for Level 3

#### 1. Import Additional Geometry

We need to import the `Circle` geometry from PhysicsNeMo:

```python
from physicsnemo.sym.geometry.primitives_2d import Circle
```

#### 2. Use the Same Wave Equation Class

We can reuse the `WaveEquation2D` class from Level 1:

```python
class WaveEquation2D(PDE):
    name = "WaveEquation2D"

    def __init__(self, c=1.0, R=1.0, alpha=1.0, beta=0.5):
        x = Symbol("x")
        y = Symbol("y")
        t = Symbol("t")
        
        input_variables = {"x": x, "y": y, "t": t}
        u = Function("u")(*input_variables)
        
        if type(c) is str:
            c = Function(c)(*input_variables)
        elif type(c) in [float, int]:
            c = Number(c)
        
        self.equations = {}
        self.equations["wave_equation"] = FIXME # Fill in: same as Level 1
        
        # Add Robin boundary condition: alpha*u + beta*(du/dn) = 0
        # For a circle centered at origin, the outward normal is (x/R, y/R)
        # So du/dn = u_x * (x/R) + u_y * (y/R)
        self.equations["robin_bc"] = FIXME # Fill in: Robin boundary condition: alpha*u + beta*(du/dn)
```

#### 3. Set Up Circular Geometry and Robin BC

```python
@physicsnemo.sym.main(config_path="conf", config_name="config_wave")
def run(cfg: PhysicsNeMoConfig) -> None:
    # Initialize wave equation with Robin BC parameters
    c = 1.0
    alpha = 1.0
    beta = 0.5
    R = 1.0  # radius
    we = WaveEquation2D(c=c, R=R, alpha=alpha, beta=beta)
    
    # Create neural network
    wave_net = instantiate_arch(
        input_keys=[Key("x"), Key("y"), Key("t")],
        output_keys=[Key("u")],
        cfg=cfg.arch.fully_connected,
    )
    
    # Create nodes
    nodes = we.make_nodes() + [wave_net.make_node(name="wave_network")]
    
    # Define circular geometry and time range
    x, y, t_symbol = Symbol("x"), Symbol("y"), Symbol("t")
    geo = Circle(center=(0, 0), radius=R)
    time_range = {t_symbol: (0, 3.0)}
    
    # Create domain
    domain = Domain()
    
    # Add initial condition with two Gaussian sources
    IC = PointwiseInteriorConstraint(
        nodes=nodes,
        geometry=geo,
        outvar={ FIXME }, # Fill in: two Gaussian sources for "u" and "u__t"
        batch_size=cfg.batch_size.IC,
        lambda_weighting={"u": 1.0, "u__t": 1.0},
        parameterization={t_symbol: 0.0},
    )
    domain.add_constraint(IC, "IC")
    
    # Add Robin boundary condition
    # The robin_bc equation is already defined in the WaveEquation2D class
    BC = PointwiseBoundaryConstraint(
        nodes=nodes,
        geometry=geo,
        outvar={ FIXME }, # Fill in: Set robin_bc to 0
        lambda_weighting={"robin_bc": 1.0},
        batch_size=cfg.batch_size.BC,
        parameterization=time_range,
    )
    domain.add_constraint(BC, "BC")
    
    # Add interior constraint
    interior = PointwiseInteriorConstraint(
        nodes=nodes,
        geometry=geo,
        outvar={ FIXME }, # Fill in: same as Level 1
        batch_size=cfg.batch_size.interior,
        parameterization=time_range,
    )
    domain.add_constraint(interior, "interior")
    
    # Create and run solver
    slv = Solver(cfg, domain)
    slv.solve()
```

1. Fill in all missing parts
2. Place the configuration file in the conf directory
3. Run the script:

In [ ]:
!python wave_l3.py

--- 

Don't forget to check out additional [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources) and join our [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack) to share your experience and get more help from the community.

---

# Licensing

Copyright © 2024 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.